In [2]:
import re
import pandas as pd
import numpy as np
from scipy import stats

# 1. Membaca Data dari Berkas Excel
excel_file = 'Blank Quiz (Responses).xlsx'
df = pd.read_excel(excel_file)

# Memfilter data khusus untuk kelas 3 TI A sesuai ketentuan tugas
df_3tia = df[df['Kelas'].str.strip() == '3 TI A'].copy()

# Mengambil kolom kuesioner yang memiliki tanda kurung siku [...]
question_cols = [c for c in df_3tia.columns if '[' in c and ']' in c]
butir = df_3tia[question_cols]

print(f"Jumlah responden (n) kelas 3 TI A: {len(butir)}")
print(f"Jumlah butir pertanyaan: {butir.shape[1]}")
print(f"Data kosong (missing data): {int(butir.isna().sum().sum())}")

# 2. Konversi Jawaban Skala Likert Menjadi Angka
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}

num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")
print(f"Sel yang gagal dikonversi: {int(num.isna().sum().sum())} (Harus 0)")

# 3. Mengelompokkan Butir Berdasarkan Dimensi
dimensi = {}
for kolom in butir.columns:
    nama = re.match(r"^(.*?) \[", kolom).group(1).strip()
    dimensi.setdefault(nama, []).append(kolom)

for nama, daftar in dimensi.items():
    print(f"- {nama}: {len(daftar)} butir")

# 4. Uji Validitas (Korelasi Pearson Butir-Total)
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)
print(f"\nn = {n} | r_tabel (alpha=0.05; df={n-2}) = {r_tabel:.3f}")

hasil_validitas = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)
    for i, kolom in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kolom], total)
        total_minus_item = total - x[kolom]
        r_koreksi, _ = stats.pearsonr(x[kolom], total_minus_item)
        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Corrected r": round(r_koreksi, 3),
            "p-value": round(p, 4),
            "Keterangan": "Valid" if r > r_tabel else "Tidak valid"
        })

tabel_validitas = pd.DataFrame(hasil_validitas)
print("\n--- TABEL VALIDITAS ---")
print(tabel_validitas.to_string())

# 5. Uji Reliabilitas (Cronbach's Alpha & Alpha Jika Butir Dihapus)
def cronbach_alpha(x):
    k = x.shape[1]
    if k <= 1:
        return np.nan
    var_sum = x.var(ddof=1).sum()
    total_var = x.sum(axis=1).var(ddof=1)
    if total_var == 0:
        return 0
    return k / (k - 1) * (1 - var_sum / total_var)

def kategori(a):
    if pd.isna(a):
        return "Tidak dapat dihitung"
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80 else "Reliabel" if a >= 0.70 else "Tidak reliabel")

hasil_reliabilitas = []
alpha_dihapus = []

for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)
    hasil_reliabilitas.append({
        "Dimensi": nama,
        "Jumlah butir": len(daftar),
        "Cronbach's alpha": round(a, 3),
        "Kategori": kategori(a)
    })
    for i, kolom in enumerate(daftar, 1):
        if len(daftar) > 2:
            x_drop = x.drop(columns=kolom)
            alpha_del = cronbach_alpha(x_drop)
            alpha_dihapus.append({
                "Dimensi": nama,
                "Butir": f"B{i}",
                "Alpha jika dihapus": round(alpha_del, 3)
            })

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)

print("\n--- TABEL RELIABILITAS ---")
print(tabel_reliabilitas.to_string())

print("\n--- TABEL ALPHA JIKA BUTIR DIHAPUS ---")
print(tabel_alpha_dihapus.to_string())

Jumlah responden (n) kelas 3 TI A: 39
Jumlah butir pertanyaan: 29
Data kosong (missing data): 0
Sel yang gagal dikonversi: 0 (Harus 0)
- Penggunaan AI pada tugas atau projek kuliah: 7 butir
- Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 7 butir
- Kebiasaan penggunaan teknologi digital: 6 butir
- Prokrastinasi akademik.: 4 butir
- Rsspon emosional: 5 butir

n = 39 | r_tabel (alpha=0.05; df=37) = 0.316

--- TABEL VALIDITAS ---
                                                         Dimensi Butir  r hitung  Corrected r  p-value   Keterangan
0                    Penggunaan AI pada tugas atau projek kuliah    B1     0.525        0.277   0.0006        Valid
1                    Penggunaan AI pada tugas atau projek kuliah    B2     0.269        0.105   0.0972  Tidak valid
2                    Penggunaan AI pada tugas atau projek kuliah    B3     0.643        0.379   0.0000        Valid
3                    Penggunaan AI pada tugas atau projek kuliah    B4     0.669        0.

/tmp/ipykernel_732/1088649700.py:30: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")
